# EvoEmu — Dyna-1-guided directed evolution (first target: KaiB)

In-silico directed evolution of a protein toward a target **conformational landscape**:

1. **Dyna-1** proposes mutations (saturation scan of predicted µs–ms exchange probability).
2. **BioEmu** scores each candidate's structural ensemble (log-likelihood ratio between two states).
3. A **genetic algorithm** keeps the best and recombines them, round after round.

**To run KaiB:** run the cells top to bottom on a GPU machine (Runpod / Lambda / Colab with a GPU).
Set `MOCK = True` in the *Parameters* cell first if you only want to rehearse the plumbing without a GPU.

## Project notes (read this first)

### How it fits together
| Piece | Role | Where |
|---|---|---|
| Genetic algorithm | selection / crossover / convergence | `protein_optimizer/genetic_algorithm.py`, `evolutionary_search.py` |
| **Dyna-1** (ESM-3 variant, sequence-only) | the *only* mutation oracle | `protein_optimizer/dyna1.py` + `scripts/dyna1_worker.py` |
| **BioEmu** | structural ensemble → LLR fitness | `protein_optimizer/bioemu.py` |
| Config | every knob, YAML + `--set key=value` overrides | `config/*.yaml`, `protein_optimizer/config.py` |

ESM-2 was removed entirely. No ESM-2 model is loaded anywhere in the search. (The structure *viewer* in the web app still calls the external ESMFold API — that is only visualisation.)

### How Dyna-1 "proposes" mutations
Dyna-1 is a per-residue classifier (`p_exchange`), not a generative model — it has **no amino-acid head**. So for each chosen position we:
1. build the 19 single-point mutants,
2. score each with Dyna-1 (mean `p_exchange` over the chain, or over `dyna1.target_positions`),
3. rank by the change vs. the parent, signed by `dyna1.direction` (`up` = more dynamic, `down` = more rigid),
4. z-score the changes across the 19 substitutions (so `dyna1.temperature` is scale-free) and sample the top-k with a softmax.

**Caveat:** nothing checks biological plausibility any more (ESM-2 used to). The search can drift toward sequences that merely look dynamic. Watch the mutation counts and the BioEmu LLR, not just the Dyna-1 signal.

### Architecture decision: Dyna-1 runs in its own virtualenv
Dyna-1 pins `numpy<2`, `biotite==0.41.2`, `transformers<4.47`; BioEmu's jax stack needs `numpy>=2`. They cannot share an interpreter (biotite 0.41.2's wheel crashes under numpy 2 with *"numpy.core.multiarray failed to import"*). Fix: `setup_vm.sh` builds `Dyna-1/.venv`, and `Dyna1MutationProposer` talks JSON-lines to a long-lived worker process there. The main process needs no torch for Dyna-1.

### Gotchas we hit on a real GPU (so you don't)
* **Dyna-1's README is out of date.** Weights are `dyna1.pt` at the **root** of `huggingface.co/gelnesr/Dyna-1`, not under `weights/`.
* `huggingface-cli` no longer exists in `huggingface_hub` 2.x — use the Python `snapshot_download` API (setup does).
* pip **reuses its wheel cache** — a rebuilt-from-source wheel can silently be the old numpy-1 build. (Moot now that Dyna-1 has its own venv.)
* Upstream `ESM_model` leaves the classifier head on CPU while the ESM-3 backbone goes to CUDA — the worker moves the whole model explicitly.
* On first load Dyna-1 downloads the open **ESM-3** base model (`esm3_sm_open_v1`). If that repo is gated for you: accept its license on huggingface.co and set `HF_TOKEN`.
* **Errors are loud on purpose.** The mutator used to fall back to random mutations if the oracle failed — a "Dyna-1-guided" run could silently be random. It now raises.
* Runpod **community-cloud** GPUs sell out between the stock read and the create, and a stopped community pod can fail to restart ("not enough free GPUs on the host"). Everything lives in git + HF, so just create a fresh pod.
* `update-pod` + `restart` on a community pod can drop the host's GPU. Prefer creating a new pod over editing a running one.

### Efficiency (already in the code)
* Candidates are de-duplicated within a round and across rounds before BioEmu (the expensive call).
* Dyna-1 scores are cached per sequence; all mutants for all chosen positions share batches; bf16 autocast on CUDA (`dyna1.half_precision`).
* Not done yet: BioEmu is still called once per sequence (it reloads the model each call — true batching needs GPU testing); prefiltering with Dyna-1 before BioEmu; fewer BioEmu samples in exploratory rounds.

### Open decisions (yours)
* **`dyna1.direction`** — `up` (default) or `down`.
* **`dyna1.target_positions`** — residues to focus Dyna-1's signal on (0-based). Unset = whole chain.
* **KaiB goal** — maximise LLR (default below) vs. hit a specific target LLR = ln(p₁/p₂) for the two states.

### KaiB
KaiB (*Synechococcus elongatus* PCC 7942, UniProt **Q79PF5**, 102 aa) is a **fold-switching** protein: it interconverts between a ground-state fold and a thioredoxin-like fold-switched state — exactly the kind of two-state landscape BioEmu's LLR is meant to capture. Sequence below was checked against UniProt on 2026-11-01.

In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
MOCK = False                 # True = synthetic Dyna-1 + BioEmu (no GPU; rehearsal only)

KAIB = ("MSPRKTYILKLYVAGNTPNSVRALKTLKNILEVEFQGVYALKVIDVLKNPQLAEEDKILA"
        "TPTLAKVLPLPVRRIIGDLSDREKVLIGLDLLYGELQDSDDF")          # UniProt Q79PF5
SEQUENCE = KAIB

TARGET_LLR = None            # None = maximise LLR; or a number = ln(p1/p2) to fit, e.g. -1.5
POPULATION = 20              # candidates per round   (start small: BioEmu is the slow part)
ROUNDS = 2                   # rounds
BIOEMU_SAMPLES = 20          # ensemble size per sequence (use 100+ for a real run)

DYNA1_DIRECTION = "up"       # "up" = more µs–ms dynamics, "down" = more rigid
DYNA1_TARGET_POSITIONS = None  # e.g. [10, 11, 12] (0-based) to focus Dyna-1 on a region; None = whole chain
DYNA1_TEMPERATURE = 1.0      # z-scored softmax sharpness; lower = greedier

REPO_URL = "https://github.com/ibrahimaasim77/EvoEmu.git"
BRANCH = "dyna1-mutation-oracle"

In [ ]:
# ── Locate (or fetch) the repo ─────────────────────────────────────────────────
import json, os, subprocess, sys, shlex
from pathlib import Path

def sh(cmd, cwd=None, check=True):
    """Run a shell command, streaming its output live."""
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, executable="/bin/bash")
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

here = Path.cwd().resolve()
ROOT = next((d for d in [here, *here.parents] if (d / "main.py").exists() and (d / "setup_vm.sh").exists()), None)
if ROOT is None:                              # e.g. a bare Jupyter/Colab session
    base = Path("/workspace") if Path("/workspace").exists() else Path.home()
    sh(f"git clone -b {BRANCH} {REPO_URL} {shlex.quote(str(base / 'EvoE'))}", check=False)
    ROOT = base / "EvoE" / "protein_optimizer"
os.chdir(ROOT)
print("repo root:", ROOT)

In [ ]:
# ── One-time setup (idempotent; ~10 min the first time on a fresh GPU box) ─────
# Builds .venv (BioEmu) and Dyna-1/.venv (Dyna-1), downloads weights, runs a mock smoke test.
if MOCK:
    print("MOCK mode: skipping GPU setup. Needs only: pip install numpy pyyaml pytest")
else:
    ready = (ROOT / ".venv/bin/python").exists() and (ROOT / "Dyna-1/model/weights/dyna1.pt").exists() \
            and (ROOT / "Dyna-1/.venv/bin/python").exists()
    if ready:
        print("Setup already done — skipping. (Delete .venv / Dyna-1 to force a rebuild.)")
    else:
        sh("bash setup_vm.sh")

In [ ]:
# ── Pre-flight: unit tests + a REAL Dyna-1 scan before spending GPU time on BioEmu ──
py = sys.executable if MOCK else str(ROOT / ".venv/bin/python")
sh(f"{py} -m pytest tests -q")
if not MOCK:
    sh(f"{py} scripts/smoke_dyna1.py {KAIB}")     # must end with: SMOKE TEST OK

In [ ]:
# ── Run the search on KaiB ─────────────────────────────────────────────────────
overrides = {
    "ga.population_size": POPULATION,
    "ga.max_generations": ROUNDS,
    "bioemu.num_samples": BIOEMU_SAMPLES,
    "dyna1.direction": DYNA1_DIRECTION,
    "dyna1.temperature": DYNA1_TEMPERATURE,
}
cmd = [str(ROOT / ".venv/bin/python") if not MOCK else sys.executable, "main.py",
       "--config", "config/evolutionary.yaml", "--sequence", SEQUENCE]
if MOCK:
    cmd.append("--mock")
if TARGET_LLR is not None:
    cmd += ["--target", str(TARGET_LLR)]
for k, v in overrides.items():
    cmd += ["--set", f"{k}={v}"]
if DYNA1_TARGET_POSITIONS:
    cmd += ["--set", f"dyna1.target_positions={json.dumps(list(DYNA1_TARGET_POSITIONS))}"]

Path("results").mkdir(exist_ok=True)
log = ROOT / "results" / "kaib_run.log"
print("running:", " ".join(shlex.quote(c) for c in cmd), "\n")
sh(" ".join(shlex.quote(c) for c in cmd) + f" 2>&1 | tee {shlex.quote(str(log))}")

In [ ]:
# ── Results: the ranked mutants from the log ───────────────────────────────────
text = (ROOT / "results" / "kaib_run.log").read_text()
i = text.find("RESULTS")
print(text[i - 70:] if i != -1 else text[-3000:])
print("\nTrajectories (if BioEmu produced them):", ROOT / "results" / "trajectories")

## Next steps
* If the smoke test passes but a run is slow: lower `BIOEMU_SAMPLES` / `POPULATION` first — BioEmu dominates wall time.
* Decide `DYNA1_DIRECTION` and `DYNA1_TARGET_POSITIONS` for KaiB (e.g. the fold-switching region), then raise `ROUNDS`.
* If you want a numeric goal instead of "maximise", set `TARGET_LLR` to ln(p₁/p₂) for the two KaiB states you care about.
* **When done on a rented GPU, stop the pod** (Runpod console, or ask Claude to). A running pod bills whether or not you use it.